# Blueprints — reproducción, con entorno de ejecución **local**

Este notebook vive en Drive y se abre en Colab, pero **debe correr en tu Mac**,
no en la nube de Google: los datos son 3,3 GB y se quedan en tu disco.

## Cómo conectarlo (una vez)

En una terminal de tu Mac:

```bash
pip install jupyter_http_over_ws
jupyter server extension enable --py jupyter_http_over_ws
jupyter notebook \
  --NotebookApp.allow_origin='https://colab.research.google.com' \
  --port=8888 --NotebookApp.port_retries=0
```

Copiá la URL con el `token=` que imprime. En Colab, arriba a la derecha:
**Conectar → Conectarse a un entorno de ejecución local**, y pegala.

Si la primera celda no encuentra tus carpetas, estás en la nube y no en tu
máquina: volvé a conectar.

In [ ]:
# 0. ¿estamos donde tienen que estar los datos?
import os, sys, subprocess, textwrap

REPO = os.path.expanduser("~/Desktop/MIR_journal")      # ajustá si lo movés
DATASETS = os.path.expanduser("~/git/Percepltion_topology/datasets")
DATA     = os.path.expanduser("~/git/Percepltion_topology/data")

ok = True
for name, p in (("repo", REPO), ("datasets", DATASETS), ("data", DATA)):
    hit = os.path.isdir(p)
    ok &= hit
    print(("  OK  " if hit else "FALTA ") + f"{name}: {p}")

if not ok:
    print(textwrap.dedent("""
    -----------------------------------------------------------------
    No encuentro las carpetas. Dos causas posibles:
      1. Estás conectado al entorno de la NUBE, no al local. Reconectá.
      2. Las rutas de arriba no son las tuyas: corregilas y reejecutá.
    -----------------------------------------------------------------"""))
else:
    print("\nEntorno local confirmado. Los datos no salen de tu disco.")

## 1. Dependencias

Se instalan en el entorno de tu Mac, no en la nube.

In [ ]:
!pip -q install numpy scipy scikit-learn pandas matplotlib wfdb pyarrow
import numpy, scipy, sklearn, pandas, matplotlib
print("python", sys.version.split()[0])
for m in (numpy, scipy, sklearn, pandas, matplotlib):
    print(f"  {m.__name__:14s} {m.__version__}")

## 2. La medición

Trece etapas, una pasada. Son horas: la etapa solar sola lee 80 archivos de
estación. `run_p` va a ser la primera corrida **completa** — `run_o` se armó
por `--restage` y le faltan `numbers.tex` y dos tablas de PM.

In [ ]:
%cd {REPO}/v8_final/repro
!python reproduce.py --data-root {DATASETS} --out run_p

## 3. Los tres diagnósticos

Responden preguntas sobre los activos, no afirmaciones del paper. Su salida va
a los apéndices.

In [ ]:
!python ecg_probe.py  --data-dir {DATASETS}/ecg/mitdb
!python pm_pooling.py --data-root {DATASETS} --out pm_pooling.json
!python pm_sets.py    --data-root {DATASETS}
!python pm_sets.py    --controls-out pm_controls.json   # los dos controles sintéticos de la tabla de pares

## 3b. Los controles de los apéndices

Cuatro apéndices y dos remarks salen de scripts que corren **después** de la
medición y leen lo que dejó: la caché de `run_p/cache/` y, algunos, sus JSON.
Cada uno escribe su criterio o su predicción (`CRITERIO.json` /
`PREDICCION.json`) **antes** de medir. Todo va a `run_p/controles/`.

Los dominios físicos tardan segundos o un par de minutos; los brazos de MNIST
dominan, decenas de minutos por script.

In [ ]:
R, W = "run_p", "run_p/controles"
os.makedirs(W, exist_ok=True)

### Control de métrica — coseno contra supremo

Tres lotes, y **en este orden**: `control_sup.py` escribe las filas de la tabla
en el orden en que se combinan las partes. Los valores no dependen del lote.
`--reference` comprueba que el brazo coseno reproduce `convergence.json`
(imprime `IDENTICO`).

In [ ]:
!python control_sup.py --data-root {DATASETS} --cache-from {R} --reference {R}/convergence.json --only "PM real" "PM simulated" "ECG real" "ECG simulated" "Battery" --out {W}/sup_a
!python control_sup.py --data-root {DATASETS} --cache-from {R} --reference {R}/convergence.json --only "UCI digits" "Spoken digits" "NOAA tides" "NSRDB solar" --out {W}/sup_b
!python control_sup.py --data-root {DATASETS} --cache-from {R} --reference {R}/convergence.json --only "MNIST" --out {W}/sup_mnist
!python control_sup.py --combine {W}/sup_a/control_sup.json {W}/sup_b/control_sup.json {W}/sup_mnist/control_sup.json --out {W}/sup_all
!cp {W}/sup_all/table_control_sup.tex ../tex/table_metric_control.tex
!cp {W}/sup_all/fig_control_sup.png   ../tex/fig_metric_control.png

!python control_tides.py --data-root {DATASETS} --reference-extra {R}/cache/conv_extra.npz --out {W}/tides
!cp {W}/tides/table_control_sup.tex ../tex/table_metric_control_tides.tex

### Lo que mide el máximo muestral

`check_boundary_quantile.py` lee por construcción los cinco pools de series
temporales de campo y simulados. Ruido y nuisance son las dos explicaciones
alternativas que el apéndice pone a prueba: solo prosa, sin tabla.

In [ ]:
!python check_one_over_n.py --data-root {DATASETS} --cache-from {R} --convergence {R}/convergence.json --out {W}/one_over_n
!python check_boundary_quantile.py --data-root {DATASETS} --cache-from {R} --out {W}/quantile
!python emit_maximum_appendix.py --one-over-n {W}/one_over_n --quantile {W}/quantile --out ../tex
!cp {W}/one_over_n/fig_one_over_n.png ../tex/

!python check_boundary_noise.py    --data-root {DATASETS} --cache-from {R} --skip-mnist --out {W}/noise
!python check_boundary_nuisance.py --data-root {DATASETS} --cache-from {R} --only "PM real" "ECG real" --out {W}/nuisance

### La clase, contra lo que no es la clase

El último par es **post hoc** y lo declara en el script, en su `CRITERIO.json`
y en el paper: el brazo de MNIST con deskew se corrió después de ver que el
brazo $\ell_2$ no pasaba el criterio del borde. Mismo criterio, mismos
sorteos, solo cambia el cociente. Recalcula además el brazo $\ell_2$: si no
reproduce `table_border_auc` (0.768 / 0.881 / 0.935 / 0.980, se asienta en
800), el brazo deskew no se lee — la celda siguiente lo verifica.

In [ ]:
!python check_nn_cover.py --data-root {DATASETS} --cache-from {R} --dimension {R}/dimension.json --out {W}/nn
!python emit_nn_cover.py --parts {W}/nn/nn_cover.json --one-over-n {W}/one_over_n/one_over_n.json --out ../tex --json-out {W}/nn_all

!python check_border_auc.py --data-root {DATASETS} --cache-from {R} --out {W}/border
!python emit_border_auc.py --parts {W}/border/border_auc.json --out ../tex --json-out {W}/border_all

!python check_border_auc_deskew.py --cache-from {R} --out {W}/border_deskew
!python emit_border_deskew.py --json {W}/border_deskew/border_auc_deskew.json --out ../tex

In [ ]:
# el brazo l2 del deskew tiene que reproducir la fila MNIST de table_border_auc
import json
raw = json.load(open(f"{W}/border_deskew/border_auc_deskew.json"))["arms"]["raw"]
ref = json.load(open(f"{W}/border_all/border_auc_all.json"))["domains"]["MNIST 28x28"]
same = raw["auc"] == ref["auc"] and raw["n_settle"] == ref["n_settle"]
print("brazo l2 == table_border_auc:", same)
if not same:
    print("NO leer el brazo deskew: el brazo de referencia no se reprodujo.")

## 4. Las diez figuras del paper

No las produce `reproduce.py`: sus emisores internos son diagnósticos y no
aparecen en el manuscrito. Éstas escriben directo en `tex/`.

In [ ]:
!python src/figure_one.py              {DATASETS} run_p ../tex/fig1_seven_phenomena.png
!python src/figure_bulk_boundary.py    run_p ../tex/fig_bulk_boundary.png
!python src/figure_digits_per_class.py run_p ../tex
!python src/figure_generators.py       {DATASETS} ../tex
!python src/figure_multisite.py compute {DATA} multisite.json
!python src/figure_multisite.py plot    multisite.json ../tex

## 4b. Las diez tablas derivadas

No las escribe una etapa: `emit_tables.py` las arma con lo que dejaron las
secciones anteriores — la corrida, los dos diagnósticos de PM, la caché
multisitio, los controles de `pm_sets.py` y el control de métrica combinado
(para la columna L∞ de `table_crossings`). Sin argumentos lee las rutas del
autor; acá van todas explícitas.

In [ ]:
!python emit_tables.py --run {R} --pooling pm_pooling.json --multisite multisite.json --sets pm_sets.json --controls pm_controls.json --control {W}/sup_all/control_sup.json --out ../tex

## 5. Adoptar la corrida y verificar

Las tablas se copian a `tex/`; después el verificador compara la **prosa**
contra la corrida, que es lo único que no está garantizado por construcción.

In [ ]:
!cp run_p/table_*.tex ../tex/
!python check_numbers.py --run run_p --tex ../tex

## 6. Compilar

Esperado: **74 páginas, 0 referencias indefinidas, 0 citas indefinidas,
0 avisos de bibtex, 0 overfull por encima de 20 pt.**

In [ ]:
%cd {REPO}/v8_final/tex
!pdflatex -interaction=nonstopmode the_geometry_of_intelligence > /dev/null
!pdflatex -interaction=nonstopmode the_geometry_of_intelligence > /dev/null
!bibtex the_geometry_of_intelligence
!pdflatex -interaction=nonstopmode the_geometry_of_intelligence > /dev/null
!pdflatex -interaction=nonstopmode the_geometry_of_intelligence | grep -E "Output written|Warning"
!python -c "import re;l=open('the_geometry_of_intelligence.log').read();print('undefined:',l.count('undefined'))"

## 7. Comparar contra la corrida del manuscrito

`--compare` distingue tres cosas que es fácil confundir: que difiera el
código, que difieran las entradas, y que un artefacto exista en una sola de
las dos corridas (una etapa agregada, no un fallo de determinismo).

In [ ]:
%cd {REPO}/v8_final/repro
!python reproduce.py --compare run_p ../../v8/run_o